# 🌴 Tropical Diseases — Clinical Knowledge Graph with LightRAG (Local Mac & Colab)

This notebook builds a **medical Knowledge Graph** from 2,893 tropical disease case reports using **LightRAG** and **Ollama** (100% offline, private, and free — NO API keys or rate limits!).

### ⚡ 2026 SOTA Model Stack (Optimized for Apple Silicon M1 & Colab):
| Component | Model | Size | Why |
| :--- | :--- | :--- | :--- |
| **LLM** | `qwen2.5:3b` | ~1.9 GB | **#1 SOTA 3B model for structured JSON extraction** — faster and more accurate than older 4B models |
| **Embedding** | `bge-m3` or `nomic-embed-text` | 274–560 MB | **Gold-standard medical embeddings** (dense + sparse retrieval, 8k context) |

### 🚀 Dual Platform Support:
- **On your Mac (DataSpell / VS Code / Jupyter)**: Runs 100% locally on Apple Silicon (M1 Metal GPU) using your `.venv`.
- **On Google Colab**: Runs on free T4 GPU.

---
## Step 1: Verify GPU

In [1]:
import sys, platform

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Running on Google Colab")
    !nvidia-smi
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU ready: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
else:
    print(f"💻 Running locally on Mac: {platform.machine()} ({platform.system()})")
    try:
        import torch
        if torch.backends.mps.is_available():
            print("✅ Apple Silicon Metal (MPS) GPU acceleration is available!")
        else:
            print("ℹ️ CPU mode active")
    except ImportError:
        print("ℹ️ Local Python environment ready (PyTorch not required for Ollama)")

💻 Running locally on Mac: arm64 (Darwin)
ℹ️ Local Python environment ready (PyTorch not required for Ollama)


---
## Step 2: Install Ollama & LightRAG

In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install zstd and Ollama on Colab
    !sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio
    print("✅ Colab dependencies & Ollama installed!")
else:
    # On Mac, check existing .venv packages
    try:
        try:
            import nest_asyncio
        except ImportError:
            import nest_asyncio2 as nest_asyncio
        import lightrag, ollama, tqdm
        print("✅ All dependencies are already installed in your local .venv!")
    except ImportError as e:
        print(f"Installing missing dependency: {e}")
        !pip install -q lightrag-hku nano-vectordb ollama tqdm nest_asyncio
        print("✅ Dependencies installed!")

✅ All dependencies are already installed in your local .venv!


---
## Step 3: Start Ollama & Pull Models (~45 seconds)

In [3]:
import os, subprocess, time, urllib.request

# Choose models:
LLM_MODEL = "qwen2.5:3b"            # SOTA 3B extraction model (~1.9 GB, fast & accurate JSON)
EMBEDDING_MODEL = "bge-m3"          # Gold standard for medical RAG (1024 dims). Or use "nomic-embed-text" (768 dims)

# 🔑 CRITICAL SPEED OPTIMIZATION:
# Keep BOTH the LLM and Embedding model loaded in RAM simultaneously!
# This eliminates model swapping/thrashing which previously caused 55-minute delays!
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "2"
os.environ["OLLAMA_NUM_PARALLEL"] = "2"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# Check if Ollama server is already running
ollama_running = False
try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
    ollama_running = True
    print("✅ Ollama server is already running!")
except Exception:
    pass

if not ollama_running:
    print("Starting Ollama server in background...")
    ollama_bin = "/usr/local/bin/ollama" if os.path.exists("/usr/local/bin/ollama") else "ollama"
    subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=os.environ
    )
    time.sleep(4)
    print("✅ Ollama server started with dual-model memory enabled!")

# Pull models (if not already present)
print(f"\nVerifying models ({LLM_MODEL} and {EMBEDDING_MODEL})...")
!ollama pull {LLM_MODEL}
!ollama pull {EMBEDDING_MODEL}

print("\n--- Installed Models ---")
!ollama list

Starting Ollama server in background...
✅ Ollama server started with dual-model memory enabled!

Verifying models (qwen2.5:3b and bge-m3)...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 5ee4f07cdb9b: 100% ▕██████████████████▏ 1.9 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling b5c0e5cf74cf: 100% ▕██████████████████▏ 7.4 KB                         
pulling 161ddde4c9cd: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling mani

---
## Step 4: Mount Google Drive (Recommended)
Saves the Knowledge Graph directly to your Google Drive so you **never lose progress** if Colab disconnects.

Skip this cell if you prefer local-only storage.

In [4]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    USE_DRIVE = True
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            STORAGE_DIR = "/content/drive/MyDrive/tropical_kg_storage"
            print("✅ Google Drive mounted. Storage:", STORAGE_DIR)
        except Exception:
            STORAGE_DIR = "/content/tropical_kg_storage"
            print("Using local Colab storage:", STORAGE_DIR)
    else:
        STORAGE_DIR = "/content/tropical_kg_storage"
else:
    STORAGE_DIR = "./tropical_kg_storage"
    print("✅ Running on Mac! KG will be saved locally to:", os.path.abspath(STORAGE_DIR))

os.makedirs(STORAGE_DIR, exist_ok=True)
print(f"📁 Storage directory: {STORAGE_DIR}")

✅ Running on Mac! KG will be saved locally to: /Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/Custom_Tropical_Project/3_LightRAG_Knowledge_Graph/tropical_kg_storage
📁 Storage directory: ./tropical_kg_storage


---
## Step 5: Locate `partA_final.jsonl` (Clean Verified Dataset)
Download and unzip `kaggle_partA` from the Kaggle link in `2_Clinical_Verification/todo.md`:
👉 https://www.kaggle.com/datasets/kisokoghan/small-multimodal-project-dataset

- **On Mac (local):** Unzip inside `Custom_Tropical_Project/2_Clinical_Verification/` — the notebook finds it automatically.
- **On Colab:** Upload `partA_final.jsonl` via the Files panel → it will be at `/content/partA_final.jsonl`.
- **On Colab with Drive:** Put `partA_final.jsonl` in your Drive → it will be at `/content/drive/MyDrive/kaggle_partA/partA_final.jsonl`.

In [5]:
import os, json

possible_paths = [
    # Mac (running from 3_LightRAG_Knowledge_Graph/)
    "../2_Clinical_Verification/kaggle_partA/partA_final.jsonl",
    # Mac (running from repo root / Datasets/)
    "./Custom_Tropical_Project/2_Clinical_Verification/kaggle_partA/partA_final.jsonl",
    # Colab — uploaded directly via Files panel
    "/content/partA_final.jsonl",
    # Colab — inside a kaggle_partA folder in Files panel
    "/content/kaggle_partA/partA_final.jsonl",
    # Colab — in Google Drive root
    "/content/drive/MyDrive/partA_final.jsonl",
    # Colab — in a kaggle_partA subfolder in Drive
    "/content/drive/MyDrive/kaggle_partA/partA_final.jsonl",
]

CASES_FILE = next((p for p in possible_paths if os.path.exists(p)), None)
if CASES_FILE is None:
    raise FileNotFoundError(
        "\u274c partA_final.jsonl not found!\n"
        "Download kaggle_partA from: https://www.kaggle.com/datasets/kisokoghan/small-multimodal-project-dataset\n"
        "Then unzip it and place partA_final.jsonl in one of the paths above."
    )

all_cases = []
with open(CASES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        all_cases.append(json.loads(line))

size_mb = os.path.getsize(CASES_FILE) / 1024 / 1024
print(f"\u2705 Found clean dataset: {CASES_FILE} ({size_mb:.1f} MB)")
print(f"   Total verified cases: {len(all_cases)}")
print(f"   Sample: {all_cases[0]['case_id']} | Disease: {all_cases[0].get('present', [])}")
print(f"   Fields: {list(all_cases[0].keys())}")

✅ Found clean dataset: ../2_Clinical_Verification/kaggle_partA/partA_final.jsonl (9.0 MB)
   Total verified cases: 1797
   Sample: PMC10018284_01 | Disease: ['Scabies']
   Fields: ['case_id', 'article_id', 'title', 'case_text', 'present', 'candidates', 'main_problem', 'extra_diagnoses', 'text_type', 'species', 'terms', 'note', 'images', 'prompt_version', 'v4_packages']


---
## Step 6: Build the Knowledge Graph 🏗️

Configure `MAX_CASES` below:
- `5` → Quick smoke test (~1 min)
- `50` → Solid demo (~8-10 min)
- `500` → Research-grade (~1.5 hrs)
- `None` → All 1,797 verified cases (~3-5 hrs)

> Each case is enriched with its confirmed diagnosis, clinical evidence quotes,
> and imaging findings before ingestion — giving LightRAG higher-quality entities to extract.

In [ ]:
import asyncio
import time
import os
import shutil
from functools import partial
from tqdm.auto import tqdm
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from lightrag.utils import EmbeddingFunc

try:
    import nest_asyncio
except ImportError:
    import nest_asyncio2 as nest_asyncio

nest_asyncio.apply()

# ========== CONFIGURATION ==========
# LLM_MODEL: "qwen2.5:3b" is SOTA for JSON extraction (faster & more accurate than older 4B models)
LLM_MODEL = "qwen2.5:3b"
# EMBEDDING_MODEL: "bge-m3" (1024-dim, medical gold-standard) or "nomic-embed-text" (768-dim, 274MB)
EMBEDDING_MODEL = "bge-m3"
EMBED_DIM = 1024 if "bge-m3" in EMBEDDING_MODEL else 768

MAX_CASES = 50                      # Set to None for all 1,797 verified cases
CONCURRENCY = 2                     # Safe parallel workers for Apple M1
# ===================================

# 1. Clean storage for a fresh start
if os.path.exists(STORAGE_DIR):
    shutil.rmtree(STORAGE_DIR)
os.makedirs(STORAGE_DIR, exist_ok=True)
print(f"🧹 Cleaned {STORAGE_DIR} for fresh start!")

# 2. Turn off thinking mode to speed up extraction
os.environ["OLLAMA_LLM_THINK"] = "false"
os.environ["OLLAMA_LLM_NUM_PREDICT"] = "2048"

# 3. Dynamic workspace to prevent duplicate document tracking issues
unique_workspace = f"local_kg_{int(time.time())}"
print(f"Using workspace: {unique_workspace}")
print(f"Using LLM: {LLM_MODEL} | Embedding: {EMBEDDING_MODEL} ({EMBED_DIM} dims)")

# 4. Initialize LightRAG
rag = LightRAG(
    working_dir=STORAGE_DIR,
    workspace=unique_workspace,
    llm_model_func=ollama_model_complete,
    llm_model_name=LLM_MODEL,
    llm_model_max_async=CONCURRENCY,
    embedding_func_max_async=CONCURRENCY,
    default_embedding_timeout=120,
    entity_extraction_use_json=True,
    enable_llm_cache=False,
    llm_model_kwargs={
        "options": {
            "think": False,       # Turn off thinking mode for direct JSON generation
            "num_predict": 2048,  # Output budget for entities
            "num_ctx": 4096,      # Context window for faster processing
        }
    },
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=4096,
        func=partial(ollama_embed.func, embed_model=EMBEDDING_MODEL)
    )
)

print("Initializing LightRAG storages...")
await rag.initialize_storages()
print("✅ Storages initialized\n")

# 5. Select cases to process
cases_to_process = all_cases[:MAX_CASES] if MAX_CASES else all_cases
total = len(cases_to_process)
print(f"Ingesting {total} cases using {LLM_MODEL} + {EMBEDDING_MODEL} with {CONCURRENCY} workers...\n")

# 6. Ingest with progress bar and per-case timing
start_total = time.time()
for i, row in enumerate(tqdm(cases_to_process, desc="Building KG", unit="case")):
    start_case = time.time()

    case_id = row["case_id"]
    title = row.get("title", "")
    present_diseases = ", ".join(row.get("present", [])) or "Unknown"
    main_problem = row.get("main_problem", "N/A")

    # Pull verbatim diagnostic evidence from verified candidates
    evidence_quotes = []
    for d, info in row.get("candidates", {}).items():
        if info.get("status") == "present" and info.get("evidence"):
            evidence_quotes.extend(info["evidence"])
    evidence_str = " ".join(evidence_quotes) if evidence_quotes else "N/A"

    # Pull imaging findings (X-ray, CT, Pathology, etc.)
    img_findings = [
        f"[{img.get('image_type', 'image')}/{img.get('image_subtype', 'finding')}]: {img.get('caption', '')}"
        for img in row.get("images", []) if img.get("caption")
    ]
    img_str = " | ".join(img_findings) if img_findings else "No imaging panels."

    # Construct structured document for LightRAG entity extraction
    case_document = (
        f"Case ID: {case_id} - {title}\n"
        f"Confirmed Diagnosis: {present_diseases}\n"
        f"Primary Clinical Problem: {main_problem}\n"
        f"Diagnostic Evidence: {evidence_str}\n"
        f"Imaging Findings: {img_str}\n\n"
        f"Clinical Narrative:\n{row.get('case_text', '')}"
    )

    await rag.ainsert(case_document, ids=case_id, file_paths=f"{case_id}.txt")

    elapsed = time.time() - start_case
    tqdm.write(f"  ✓ {case_id} ({present_diseases}) indexed in {elapsed:.1f}s")

total_time = time.time() - start_total
print(f"\n{'='*60}")
print(f"✅ Knowledge Graph built! {total} cases in {total_time/60:.1f} minutes")
print(f"   Avg: {total_time/total:.1f}s per case")
print(f"   Saved to: {STORAGE_DIR}")
print(f"{'='*60}")

/Users/apple/VGU/Year 4 2026-2027/Project_Dr. Tran Duc Khanh/Datasets/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧹 Cleaned ./tropical_kg_storage for fresh start!
Using workspace: local_kg_1791022666
Using LLM: qwen2.5:3b | Embedding: bge-m3 (1024 dims)


INFO: [local_kg_1791022666] Created new empty graph file: ./tropical_kg_storage/local_kg_1791022666/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './tropical_kg_storage/local_kg_1791022666/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './tropical_kg_storage/local_kg_1791022666/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': './tropical_kg_storage/local_kg_1791022666/vdb_chunks.json'} 0 data
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=2, timeout=240
INFO:  - keyword: None/None, host=None, max_async=2, timeout=240
INFO:  - query: None/None, host=None, max_async=2, timeout=240
INFO:  - vlm: None/None, host=None, max_async=2, timeout=240
INFO: [local_kg_1791022666] Process 7638 KV load full_docs with 0 records
INFO: [local_kg_1791022666]

Initializing LightRAG storages...
✅ Storages initialized

Ingesting 50 cases using qwen2.5:3b + bge-m3 with 2 workers...



Building KG:   0%|          | 0/50 [00:00<?, ?case/s]INFO: Processing 1 document(s)
INFO: Parsing (native): PMC10018284_01
INFO: Extracting stage 1/1: PMC10018284_01.txt
INFO: Processing d-id: PMC10018284_01
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: PMC10018284_01
INFO: extract LLM func: 2 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: default:extract:9eb1b165df6acff405fb04316dc5b911
INFO:  == LLM cache == saving: default:extract:1cc6bb31642cfcedc5194a416294256b
INFO:  == LLM cache == saving: default:extract:c72a009604faa7d220355f02ec7fbfba
INFO: Chunk 1 of 2 extracted 5 Ent + 4 Rel PMC10018284_01-chunk-001
INFO:  == LLM cache == saving: default:extract:8d3d4d59bcd8949300646c867c95187c
INFO: Chunk 2 of 2 extracted 4 Ent + 2 Rel PMC10018284_01-chunk-000
INFO: Merging stage 1/1: PMC10018284_01.txt
INFO: Phase 0: Writing recovery indexes for PMC10018284_01: 9 candidate entities, 6 candidate 

...

  ✓ PMC10034413_01 (Foodborne Trematodes) indexed in 243.6s


INFO:  == LLM cache == saving: default:extract:e4b3577a0f3b6881d4e4de79b7a937a3
INFO:  == LLM cache == saving: default:extract:2afc918ad5729e1ddea14d8a39552cd7
INFO: Chunk 1 of 2 extracted 5 Ent + 3 Rel PMC10038010_01-chunk-001
INFO:  == LLM cache == saving: default:extract:0d388b9ac44bdd9f4d588a11fcbfcc98
INFO:  == LLM cache == saving: default:extract:83fa4dd9425e1e0f62c6d9ef73f834da
INFO: Chunk 2 of 2 extracted 13 Ent + 3 Rel PMC10038010_01-chunk-000
INFO: Merging stage 1/1: PMC10038010_01.txt
INFO: Phase 0: Writing recovery indexes for PMC10038010_01: 18 candidate entities, 6 candidate relations
INFO: Phase 1: Processing 18 entities from PMC10038010_01 (async: 4)
INFO: Merged: `Computed tomography (CT) scan` | 1+1
INFO: Phase 2: Processing 6 relations from PMC10038010_01 (async: 4)
INFO: Upserting relation VDB: `Hydrocephalus->Male, 26 years old`
INFO: Upserting relation VDB: `intensive care unit (ICU)->patient`
INFO: Upserting relation VDB: `Male, 26 years old->Tuberculous Meningit

  ✓ PMC10038010_01 (Tuberculosis) indexed in 153.4s


INFO:  == LLM cache == saving: default:extract:f33482bed30e1855891109c043f66a1e
INFO:  == LLM cache == saving: default:extract:caa5f938e1cd9144b83c8916339ad0ca
INFO: Chunk 1 of 1 extracted 6 Ent + 1 Rel PMC10071914_01-chunk-000
INFO: Merging stage 1/1: PMC10071914_01.txt
INFO: Phase 0: Writing recovery indexes for PMC10071914_01: 6 candidate entities, 1 candidate relations
INFO: Phase 1: Processing 6 entities from PMC10071914_01 (async: 4)
INFO: Phase 2: Processing 1 relations from PMC10071914_01 (async: 4)
INFO: Upserting relation VDB: `doxycycline->woman`
INFO: Phase 2 relation processing completed for PMC10071914_01: edges=1 added_entities=0
INFO: Completed merging: 6 entities, 0 extra entities, 1 relations
INFO: [local_kg_1791022666] entities flush: embedding 6 vectors in 1 batch(es) (batch_num=10)
INFO: [local_kg_1791022666] relationships flush: applied 0 deferred deletes (1 queued, 0 awaiting save)
INFO: [local_kg_1791022666] relationships flush: embedding 1 vectors in 1 batch(es

  ✓ PMC10071914_01 (Scrub Typhus) indexed in 71.7s


---
## Step 7: Query the Knowledge Graph 🔍

In [ ]:
test_query = "What are the common symptoms, lab abnormalities, and risk factors for Dengue fever and Tuberculosis in tropical patients?"
print(f"Query: {test_query}\n")

response = await rag.aquery(test_query, param=QueryParam(mode="hybrid"))

print("=" * 60)
print("LightRAG Diagnostic Response (Hybrid Mode)")
print("=" * 60)
print(response)

### Try More Queries
Change the query below and re-run to explore the Knowledge Graph.

In [ ]:
# Try different queries!
my_query = "What imaging findings distinguish pulmonary tuberculosis from bacterial pneumonia?"

response = await rag.aquery(my_query, param=QueryParam(mode="hybrid"))
print(response)

---
## Step 8: Download Knowledge Graph to Your Mac
Downloads `tropical_kg_storage.zip` to your computer.
Unzip it into your local `Datasets/tropical_kg_storage/` folder to use offline!

In [ ]:
import os, shutil, sys

zip_path = "./tropical_kg_storage"
shutil.make_archive(zip_path, 'zip', STORAGE_DIR)
print(f"✅ Knowledge Graph archived to: {zip_path}.zip")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("Downloading to browser...")